In [1]:
from __future__ import annotations

import re
from io import StringIO
from pathlib import Path

import pandas as pd
import requests


GROUPS = list("ABCDEFGHIJKL")

HOST_TEAMS = {
    "Mexico",
    "Canada",
    "United States",
}

TEAM_NAME_MAP = {
    "Czech Republic": "Czechia",
    "Korea Republic": "South Korea",
    "USA": "United States",
    "Türkiye": "Turkey",
    "Curaçao": "Curacao",
    "Côte d'Ivoire": "Ivory Coast",
}

MATCH_DATE_RANGES = [
    (1, 2, "2026-06-11"),
    (3, 4, "2026-06-12"),
    (5, 8, "2026-06-13"),
    (9, 12, "2026-06-14"),
    (13, 16, "2026-06-15"),
    (17, 20, "2026-06-16"),
    (21, 24, "2026-06-17"),
    (25, 28, "2026-06-18"),
    (29, 32, "2026-06-19"),
    (33, 36, "2026-06-20"),
    (37, 40, "2026-06-21"),
    (41, 44, "2026-06-22"),
    (45, 48, "2026-06-23"),
    (49, 54, "2026-06-24"),
    (55, 60, "2026-06-25"),
    (61, 66, "2026-06-26"),
    (67, 72, "2026-06-27"),
]


def read_html_tables(url: str) -> list[pd.DataFrame]:
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    return pd.read_html(StringIO(response.text))


def clean_team_name(team: object) -> str:
    team_name = str(team).strip()
    team_name = re.sub(r"\[.*?\]", "", team_name)
    team_name = team_name.replace("(H)", "").strip()
    team_name = TEAM_NAME_MAP.get(team_name, team_name)
    return team_name


def extract_match_number(value: object) -> int | None:
    match = re.search(r"Match\s+(\d+)", str(value))
    if match is None:
        return None

    return int(match.group(1))


def infer_match_date(match_number: int) -> str:
    for start_number, end_number, match_date in MATCH_DATE_RANGES:
        if start_number <= match_number <= end_number:
            return match_date

    raise ValueError(f"Número de partida fora do intervalo esperado: {match_number}")


def is_match_table(table: pd.DataFrame) -> bool:
    columns = list(table.columns)

    if len(columns) != 3:
        return False

    middle_column = str(columns[1])
    return extract_match_number(middle_column) is not None


def extract_matches_from_group(group: str) -> list[dict[str, object]]:
    url = f"https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_Group_{group}"
    tables = read_html_tables(url)

    matches = []

    for table in tables:
        if not is_match_table(table):
            continue

        columns = list(table.columns)

        home_team = clean_team_name(columns[0])
        away_team = clean_team_name(columns[2])
        match_number = extract_match_number(columns[1])

        if match_number is None:
            continue

        match_date = infer_match_date(match_number)
        neutral = 0 if home_team in HOST_TEAMS or away_team in HOST_TEAMS else 1

        matches.append(
            {
                "match_number": match_number,
                "match_date": match_date,
                "home_team": home_team,
                "away_team": away_team,
                "group": group,
                "stage": "group",
                "tournament": "FIFA World Cup",
                "neutral": neutral,
            }
        )

    return matches


def build_world_cup_schedule() -> pd.DataFrame:
    rows = []

    for group in GROUPS:
        group_matches = extract_matches_from_group(group)
        print(f"Grupo {group}: {len(group_matches)} jogos extraídos")
        rows.extend(group_matches)

    schedule_data = pd.DataFrame(rows)
    schedule_data = schedule_data.sort_values("match_number").reset_index(drop=True)

    expected_matches = set(range(1, 73))
    extracted_matches = set(schedule_data["match_number"].tolist())
    missing_matches = sorted(expected_matches - extracted_matches)
    duplicated_matches = schedule_data[
        schedule_data["match_number"].duplicated(keep=False)
    ]["match_number"].tolist()

    if missing_matches:
        raise ValueError(f"Partidas ausentes: {missing_matches}")

    if duplicated_matches:
        raise ValueError(f"Partidas duplicadas: {duplicated_matches}")

    return schedule_data


def main() -> None:
    output_path = Path("data/raw/world_cup_2026_schedule.csv")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    schedule_data = build_world_cup_schedule()
    schedule_data.to_csv(output_path, index=False)

    print("\nArquivo gerado:")
    print(output_path)
    print("\nPrévia:")
    print(schedule_data.head(10))
    print("\nDimensão:")
    print(schedule_data.shape)


if __name__ == "__main__":
    main()

Grupo A: 6 jogos extraídos
Grupo B: 6 jogos extraídos
Grupo C: 6 jogos extraídos
Grupo D: 6 jogos extraídos
Grupo E: 6 jogos extraídos
Grupo F: 6 jogos extraídos
Grupo G: 6 jogos extraídos
Grupo H: 6 jogos extraídos
Grupo I: 6 jogos extraídos
Grupo J: 6 jogos extraídos
Grupo K: 6 jogos extraídos
Grupo L: 6 jogos extraídos

Arquivo gerado:
data\raw\world_cup_2026_schedule.csv

Prévia:
   match_number  match_date      home_team               away_team group  \
0             1  2026-06-11         Mexico            South Africa     A   
1             2  2026-06-11    South Korea                 Czechia     A   
2             3  2026-06-12         Canada  Bosnia and Herzegovina     B   
3             4  2026-06-12  United States                Paraguay     D   
4             5  2026-06-13          Haiti                Scotland     C   
5             6  2026-06-13      Australia                  Turkey     D   
6             7  2026-06-13         Brazil                 Morocco     C   
7    